In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

tavily_api_key = os.getenv("TAVILY_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if tavily_api_key:
    os.environ["TAVILY_API_KEY"] = tavily_api_key
if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key

try:
    from langchain_groq import ChatGroq
except ImportError:
    ChatGroq = None
    print("Package missing: install with `pip install langchain-groq`.")

model = None
if ChatGroq and groq_api_key:
    model = ChatGroq(model="qwen/qwen3-32b")
    print("ChatGroq model initialized.")
else:
    print("GROQ_API_KEY missing or langchain-groq not installed; model not initialized.")

c:\Users\arnab\Desktop\Folders\Langchain\Langchain_Nayak\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
import os

try:
    from langchain_tavily import TavilySearch
except ImportError:
    TavilySearch = None
    print("Package missing: install with `pip install langchain-tavily`.")

if TavilySearch and os.getenv("TAVILY_API_KEY"):
    tool = TavilySearch(max_results=5, topic="general")
    results = tool.invoke("What is the current AI news?")
    print(results)
else:
    print("TAVILY_API_KEY missing or langchain-tavily not installed; search skipped.")

{'query': 'What is the current ai news',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates',
   'title': 'Latest AI News and AI Breakthroughs that Matter Most: 2026 & 2025',
   'content': 'The Latest AI Breakthroughs, News, and Updates · Atlassian Cuts 1,600 Jobs in Pivot to AI · Meta Announces Four New In-House AI Chips to Reduce Reliance on',
   'score': 0.68697983,
   'raw_content': None},
  {'url': 'https://www.artificialintelligence-news.com/',
   'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide.',
   'score': 0.68344116,
   'raw_content': None},
  {'url': 'https://www.wsj.com/tech/ai?gaa_at=eafs&gaa_n=AWEtsqfGBux42fXtVo-ZDGqDkVDazsbQ2za9-yi6SvHq3Em75bJeTt1J8V96&gaa_ts=69c58fa1&gaa_sig=GE0KjjeQSq0IrqEgiCV48ml

In [ ]:
import ast
import operator as op
from langchain.tools import tool

_ALLOWED_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.Mod: op.mod,
    ast.USub: op.neg,
}


def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Only numeric arithmetic expressions are allowed.")


@tool("calculator", description="Performs arithmetic calculations. Use this for math problems.")
def calc(expression: str) -> str:
    """Evaluate numeric arithmetic expressions safely."""
    tree = ast.parse(expression, mode="eval")
    return str(_safe_eval(tree.body))


print(calc.invoke("2 * (3 + 4)"))